In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "clean").exists() else Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

# 1. 기업개요_최종: 회사 마스터 테이블 (crno가 유니크 키)
corp = pd.read_csv(CLEAN_DIR / "기업개요_최종_보강.csv", dtype=str, encoding="utf-8-sig")
corp = corp.drop(columns=["Unnamed: 0"])
master = corp.set_index("crno")
MASTER_COLS = ["corpNm", "corpEnsnNm", "enpBsadr", "enpEstbDt", "상장여부"]

afil = pd.read_csv(CLEAN_DIR / "계열회사_전처리.csv", dtype=str, encoding="utf-8-sig")
sub = pd.read_csv(CLEAN_DIR / "종속기업_정리_보강.csv", dtype=str, encoding="utf-8-sig")

# 이름 보강용 통합 lookup: 기업개요_최종(우선) + 계열회사_전처리의 afilCmpyCrno->afilCmpyNm(대체)
name_map = master["corpNm"].to_dict()
fallback_names = (
    afil.dropna(subset=["afilCmpyCrno", "afilCmpyNm"])
    .drop_duplicates("afilCmpyCrno")
    .set_index("afilCmpyCrno")["afilCmpyNm"]
)
for crno, name in fallback_names.items():
    name_map.setdefault(crno, name)

def attach_master(df, id_col, prefix):
    sub_master = master[MASTER_COLS].add_prefix(f"{prefix}_")
    out = df.merge(sub_master, left_on=id_col, right_index=True, how="left")
    had_official = out[f"{prefix}_corpNm"].notna()
    fb = out[id_col].map(name_map)
    out[f"{prefix}_corpNm"] = out[f"{prefix}_corpNm"].fillna(fb)
    out[f"{prefix}_name_source"] = pd.NA
    out.loc[had_official, f"{prefix}_name_source"] = "기업개요_최종"
    out.loc[(~had_official) & out[f"{prefix}_corpNm"].notna(), f"{prefix}_name_source"] = "계열회사_전처리(fallback)"
    return out

# 2. 계열회사_전처리: crno(모기업) - afilCmpyCrno(계열사), 양쪽 다 ID 보유 -> 양쪽 다 ID 매칭
afil_out = afil.copy()
afil_out["relation_type"] = "계열회사"
afil_out = afil_out.rename(columns={"crno": "source_crno", "afilCmpyCrno": "target_crno", "afilCmpyNm": "target_name_raw"})
afil_out = attach_master(afil_out, "source_crno", "source")
afil_out = attach_master(afil_out, "target_crno", "target")

# 계열망(계열회사_전처리에 한 번이라도 등장하는 crno) 소속 여부
affiliate_network_ids = set(afil["crno"]) | set(afil["afilCmpyCrno"])

# 3. 종속기업_정리: crno(모기업)만 ID 보유, 종속기업 자체 ID는 없음 -> 모기업 쪽만 ID 매칭 + 이름 보강
sub_out = sub.copy()
sub_out["relation_type"] = "종속기업"
sub_out = sub_out.rename(columns={"crno": "source_crno", "sbrdEnpNm": "target_name_raw"})
sub_out["target_crno"] = pd.NA
sub_out = attach_master(sub_out, "source_crno", "source")

# 계열사 관계망을 거치지 않고 바로 종속기업으로 이어지는 경우 표시
sub_out["direct_to_subsidiary"] = ~sub_out["source_crno"].isin(affiliate_network_ids)

# 4. 공통 스키마로 통합 (세로 결합)
common_cols = [
    "relation_type", "source_crno", "source_corpNm", "source_name_source", "source_corpEnsnNm", "source_enpBsadr", "source_enpEstbDt", "source_상장여부",
    "target_crno", "target_name_raw", "target_corpNm", "target_name_source", "target_corpEnsnNm", "target_enpBsadr", "target_enpEstbDt", "target_상장여부",
    "direct_to_subsidiary",
]

afil_final = afil_out.reindex(columns=common_cols)
sub_final = sub_out.reindex(columns=common_cols)
sub_final["sbrdEnpMainBizCtt"] = sub_out["sbrdEnpMainBizCtt"].values
sub_final["domestic"] = sub_out["domestic"].values
sub_final["name_norm"] = sub_out["name_norm"].values

merged = pd.concat([afil_final, sub_final], ignore_index=True, sort=False)

merged.to_csv(CLEAN_DIR / "지식그래프_통합_ID매칭.csv", index=False, encoding="utf-8-sig")

In [2]:
import html
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "clean").exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

# ============================================================
# 0. 로드
# ============================================================
corp = pd.read_csv(CLEAN_DIR / "기업개요_최종_보강.csv", dtype=str, encoding="utf-8-sig").drop(columns=["Unnamed: 0"])
afil = pd.read_csv(CLEAN_DIR / "계열회사_전처리.csv", dtype=str, encoding="utf-8-sig")
sub = pd.read_csv(CLEAN_DIR / "종속기업_정리_보강.csv", dtype=str, encoding="utf-8-sig")

# 종속회사 주소는 정리본에서 삭제됐으므로 원본에서 복구
sub_raw = pd.read_csv(RAW_DIR / "종속기업_원본.csv", dtype=str, encoding="utf-8-sig")

# 업종(sicNm)은 기업개요_최종에 없으므로 raw에서 보충
corp_raw = pd.concat(
    [pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in sorted(RAW_DIR.glob("기업개요_*.csv"))],
    ignore_index=True,
)
sic_map = (
    corp_raw.assign(sic=corp_raw["sicNm"].fillna(corp_raw["enpMainBizNm"]))
    .dropna(subset=["sic"])
    .drop_duplicates("crno")
    .set_index("crno")["sic"]
    .to_dict()
)


# ============================================================
# 1. 공통 유틸 (텍스트 정리 / 지역 추출 / crno 조회)
# ============================================================
def clean_text(s):
    s = s.map(lambda x: html.unescape(x) if isinstance(x, str) else x)
    s = s.str.replace("&cr;", " ", regex=False)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


SIDO = {
    "서울": "서울", "부산": "부산", "대구": "대구", "인천": "인천", "광주": "광주",
    "대전": "대전", "울산": "울산", "세종": "세종", "경기": "경기", "강원": "강원",
    "충청북": "충북", "충북": "충북", "충청남": "충남", "충남": "충남",
    "전라북": "전북", "전북": "전북", "전라남": "전남", "전남": "전남",
    "경상북": "경북", "경북": "경북", "경상남": "경남", "경남": "경남", "제주": "제주",
}
SIDO_RE = re.compile("|".join(sorted(SIDO, key=len, reverse=True)))


def to_region(addr):
    """주소 문자열 -> 시/도 축약명"""
    if not isinstance(addr, str) or not addr.strip() or addr in ("-", "확인할수없음"):
        return pd.NA
    m = SIDO_RE.search(addr)
    return SIDO[m.group()] if m else pd.NA


# 이름 lookup: 기업개요_최종 우선, 계열회사_전처리(afilCmpyCrno -> afilCmpyNm) fallback
name_map = corp.set_index("crno")["corpNm"].to_dict()
for crno, nm in (
    afil.dropna(subset=["afilCmpyCrno", "afilCmpyNm"])
    .drop_duplicates("afilCmpyCrno")
    .set_index("afilCmpyCrno")["afilCmpyNm"]
    .items()
):
    name_map.setdefault(crno, nm)

addr_map = corp.dropna(subset=["enpBsadr"]).set_index("crno")["enpBsadr"].to_dict()


def enrich(crno_series, prefix):
    """crno 시리즈 -> 회사명/주소/지역/업종 컬럼 묶음"""
    addr = crno_series.map(addr_map)
    return pd.DataFrame({
        f"{prefix}_corpNm": crno_series.map(name_map),
        f"{prefix}_region": addr.map(to_region),
        f"{prefix}_addr": addr,
        f"{prefix}_sicNm": crno_series.map(sic_map),
    }, index=crno_series.index)


# ============================================================
# 2. 종속회사 주소 복구 (03_subsid_clean 과 동일 규칙)
# ============================================================
sr = sub_raw[["crno", "sbrdEnpNm", "sbrdEnpadr"]].copy()
for c in ["sbrdEnpNm", "sbrdEnpadr"]:
    sr[c] = clean_text(sr[c])

adr = sr["sbrdEnpadr"].fillna("").str.strip()
bad = adr.isin(["상동", "-", ""]) | adr.str.match(r"^\(?상\d*\)?$")
sr["sbrdEnpadr"] = sr["sbrdEnpadr"].where(~bad).groupby(sr["crno"]).ffill()

sub_addr_map = (
    sr.dropna(subset=["sbrdEnpadr"])
    .drop_duplicates(["crno", "sbrdEnpNm"])
    .set_index(["crno", "sbrdEnpNm"])["sbrdEnpadr"]
)


def sub_addr(df):
    key = pd.MultiIndex.from_arrays([df["crno"], df["sbrdEnpNm"]])
    return pd.Series(sub_addr_map.reindex(key).to_numpy(), index=df.index)


def sub_block(df):
    """종속회사 컬럼 묶음"""
    addr = sub_addr(df)
    return pd.DataFrame({
        "subsidiary_name": df["sbrdEnpNm"],
        "subsidiary_region": addr.map(to_region),
        "subsidiary_addr": addr,
        "subsidiary_bizCtt": df["sbrdEnpMainBizCtt"],
        "domestic": df["domestic"],
        "name_norm": df["name_norm"],
    }, index=df.index)


# ============================================================
# 3. 경우 1/2/3 구성
# ============================================================
affiliate_network_ids = set(afil["crno"]) | set(afil["afilCmpyCrno"])
parents_of = afil.groupby("afilCmpyCrno")["crno"].apply(lambda s: sorted(set(s))).to_dict()

EMPTY_AFIL = ["affiliate_crno", "affiliate_corpNm", "affiliate_region", "affiliate_addr", "affiliate_sicNm"]
EMPTY_SUB = ["subsidiary_name", "subsidiary_region", "subsidiary_addr", "subsidiary_bizCtt", "domestic", "name_norm"]

# --- 경우1: 모기업 - 계열회사 (같은 crno 쌍이 표기만 다르게 중복된 행 제거) ---
a = afil.drop_duplicates(["crno", "afilCmpyCrno"]).reset_index(drop=True)
print(f"계열회사 표기중복 제거: {len(afil):,} -> {len(a):,} ({len(afil) - len(a)}건)")

case1 = pd.concat([
    pd.DataFrame({"case": 1, "top_crno": a["crno"]}),
    enrich(a["crno"], "top"),
    pd.DataFrame({"affiliate_crno": a["afilCmpyCrno"]}),
    enrich(a["afilCmpyCrno"], "affiliate"),
    pd.DataFrame(pd.NA, index=a.index, columns=EMPTY_SUB),
], axis=1)

# --- 경우2: 모기업 - 종속회사 (계열망 밖에서 바로 연결) ---
s2 = sub[~sub["crno"].isin(affiliate_network_ids)].reset_index(drop=True)
case2 = pd.concat([
    pd.DataFrame({"case": 2, "top_crno": s2["crno"]}),
    enrich(s2["crno"], "top"),
    pd.DataFrame(pd.NA, index=s2.index, columns=EMPTY_AFIL),
    sub_block(s2),
], axis=1)

# --- 경우3: 모기업 - 계열회사 - 종속회사 ---
# 종속회사의 모기업이 계열망 소속이면, 그 회사를 계열사로 등록한 회사들이 최상위 모기업
s3 = sub[sub["crno"].isin(affiliate_network_ids)].reset_index(drop=True)
tops = s3["crno"].map(lambda c: parents_of.get(c, []))


def join_uniq(values):
    """여러 모기업의 값을 중복 없이 ';' 로 연결"""
    return ";".join(dict.fromkeys(v for v in values if pd.notna(v))) or pd.NA


case3 = pd.concat([
    pd.DataFrame({
        "case": 3,
        "top_crno": tops.map(join_uniq),
        "top_corpNm": tops.map(lambda l: join_uniq(name_map.get(c, c) for c in l)),
        "top_region": tops.map(lambda l: join_uniq(to_region(addr_map.get(c)) for c in l)),
        "top_addr": tops.map(lambda l: join_uniq(addr_map.get(c) for c in l)),
        "top_sicNm": tops.map(lambda l: join_uniq(sic_map.get(c) for c in l)),
        "affiliate_crno": s3["crno"],
    }),
    enrich(s3["crno"], "affiliate"),
    sub_block(s3),
], axis=1)

# ============================================================
# 4. 통합 + 정제 + 저장
# ============================================================
COLS = [
    "case",
    "top_crno", "top_corpNm", "top_region", "top_addr", "top_sicNm",
    "affiliate_crno", "affiliate_corpNm", "affiliate_region", "affiliate_addr", "affiliate_sicNm",
    "subsidiary_name", "subsidiary_region", "subsidiary_addr", "subsidiary_bizCtt",
    "domestic", "name_norm",
]
merged = pd.concat([case1, case2, case3], ignore_index=True).reindex(columns=COLS)

before = len(merged)
merged = merged.drop_duplicates(["case", "top_crno", "affiliate_crno", "subsidiary_name"])
print(f"통합 후 중복 제거: {before:,} -> {len(merged):,} ({before - len(merged)}건)")

# crno만 있고 회사를 특정할 정보가 전혀 없는 행 제거
before, n_ids = len(merged), merged.loc[merged["top_corpNm"].isna(), "top_crno"].nunique()
merged = merged[merged["top_corpNm"].notna()].reset_index(drop=True)
print(f"모기업 미상 행 제거: {before:,} -> {len(merged):,} ({before - len(merged):,}건, 모기업 {n_ids:,}개사)")

out_path = CLEAN_DIR / "모기업_계열사_종속기업_통합.csv"
merged.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {out_path.relative_to(PROJECT_ROOT)} ({len(merged):,}행 × {len(merged.columns)}컬럼)")
print(merged["case"].value_counts().sort_index().to_string())

계열회사 표기중복 제거: 9,601 -> 9,513 (88건)
통합 후 중복 제거: 23,240 -> 23,240 (0건)
모기업 미상 행 제거: 23,240 -> 17,502 (5,738건, 모기업 1,338개사)

저장 완료: data\clean\모기업_계열사_종속기업_통합.csv (17,502행 × 17컬럼)
case
1    9513
2    1934
3    6055
